# B1.6 · Securing the developers' agents

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *Security of AI*

---

**Risk.** Secrets in prompts and transcripts; CI credentials reachable from an agent session.

**Control.** Scan the agent footprint the way you scan the repo.

**This lab.** Find live secrets in your own agent's footprint.

| | |
|---|---|
| Open-source tooling | gitleaks, TruffleHog |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B1.6"))

Developers' coding agents run with the developer's credentials, on the developer's machine, against the whole monorepo. That is a production identity in an unmanaged environment.

In [ ]:
from cybercommons import planes, sandbox
W = planes.Tool

dev_agent = planes.Manifest("ide-coding-agent", [
    W("read_file"),
    W("write_file", writes=True, scope="project"),
    W("run_shell",  writes=True, scope="tenant", reversible=False),
    W("git_push",   writes=True, scope="project", reversible=False),
], rung="L2.5")

b = dev_agent.blast_radius()
print("blast radius:", b["total"], b["per_tool"])
for p in dev_agent.rung_check():
    print("⚠", p)

Now the containment that is actually deployable on a laptop, without asking developers to accept a slower loop.

In [ ]:
box = sandbox.Sandbox(
    egress=sandbox.EgressPolicy(allow_hosts={"api.github.com", "registry.npmjs.org"}),
    paths=sandbox.PathGuard(workspace="/work/repo"),
    tools=sandbox.ToolPolicy(allow={"read_file", "write_file"},
                             require_approval={"run_shell", "git_push"}))
for tool, target in [("read_file", "/work/repo/src/a.py"),
                     ("read_file", "/work/repo/../../.aws/credentials"),
                     ("http_get",  "https://exfil.example.com/x"),
                     ("git_push",  "")]:
    print(box.call(tool, target))

### Expect

The unconstrained dev agent scores a high blast radius and flags irreversible ungated tools. The sandboxed version still allows normal editing while refusing the credential read, the unlisted host and the ungated push.

### Your turn

The honest constraint here is developer tolerance. Which single control would you ship first if you were only allowed one, and what is your evidence that it would survive a week?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B1.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*